In [120]:
from data_engineering.flights.flight_type import get_flights, get_flights_from_iceberg
from datetime import date,timedelta
import polars as pl

start_date = date(2025,7,1)
end_date = date(2025,8,1)
df_flights = pl.DataFrame()
for target_date in [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]:
    df_flights = pl.concat([df_flights, get_flights(target_date)], how="vertical")
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""010058""","""SUBNP""","""SU-BNP""",2025-07-01 15:14:19.210,"""HEAL""",2025-07-01 15:31:18.240,"""HEOC""",false,false,false,false,"""GLF4""",null,"""GULFSTREAM 4""","""A2"""
"""0100a2""","""MSR551""","""SU-GCE""",2025-07-01 13:09:30.800,"""HE30""",2025-07-01 13:28:17.190,"""HECA""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR542""","""SU-GCE""",2025-07-01 18:23:43.610,"""HE42""",2025-07-01 20:25:19.410,"""LTFM""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR543""","""SU-GCE""",2025-07-01 21:48:55.030,"""LTFM""",2025-07-01 23:21:37.740,"""HEAZ""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a3""","""MSR552""","""SU-GCF""",2025-07-01 07:38:19.150,"""HECA""",2025-07-01 07:49:18.290,"""HE30""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e94c8a""","""BOV795""",null,2025-07-31 18:42:42.730,"""AR-0583""",2025-07-31 19:13:21.180,"""PY-0033""",null,null,null,null,null,null,null,"""A2"""
"""e94c8e""","""BOV767""","""CP-3214""",2025-07-31 02:25:36.910,"""KMIA""",2025-07-31 03:57:41.260,"""JM-0016""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""e94c8f""","""BOV736""","""CP-3215""",2025-07-31 10:44:41.510,"""SLLJ""",2025-07-31 15:39:38.710,"""SBGR""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""


In [121]:
df_flights = df_flights.filter((pl.col("ladd") == True) | (pl.col("pia") == True))

In [122]:
df_flights = df_flights.with_columns(
    pl.concat_str(
        [
            pl.col("icao"),
            pl.col("takeoff_time").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ],
        separator="_",
    ).alias("flight_id")
)

In [126]:
df_flights = df_flights.select("flight_id", "icao", "takeoff_time", "landing_time", "landing_airport_ident", "takeoff_airport_ident", "ladd", "pia", "category")

In [127]:
next_ids = (
    df_flights.join(
        df_flights.select(
            pl.col("flight_id").alias("next_flight_id"),
            pl.col("icao").alias("next_icao"),
            pl.col("pia").alias("next_pia"),
            pl.col("category").alias("next_category"),
            "takeoff_airport_ident",
            pl.col("takeoff_time").alias("next_takeoff_time"),
        ),
        left_on="landing_airport_ident",
        right_on="takeoff_airport_ident",
        how="left",
    )
    .filter(
        (pl.col("next_takeoff_time") > pl.col("landing_time"))
        & (
            (pl.col("next_icao") == pl.col("icao"))
            | (
                pl.col("next_pia")
                & pl.col("category").is_not_null()
                & pl.col("next_category").is_not_null()
                & (pl.col("next_category") == pl.col("category"))
            )
        )
    )
    .group_by("flight_id")
    .agg(
        pl.col("next_flight_id")
        .unique()
        .alias("next_flight_ids")
    )
)

df = (
    df_flights
    .join(next_ids, on="flight_id", how="left")
    .with_columns(
        pl.col("next_flight_ids")
        .fill_null(pl.lit([], dtype=pl.List(pl.String)))
    )
)

In [128]:
import networkx as nx

G = nx.DiGraph()

# Add all flights as nodes
for row in df.iter_rows(named=True):
    G.add_node(
        row["flight_id"],
        icao=row["icao"],
        takeoff_time=row["takeoff_time"],
        landing_time=row["landing_time"],
        takeoff_airport_ident=row["takeoff_airport_ident"],
        landing_airport_ident=row["landing_airport_ident"],
        ladd=row["ladd"],
        pia=row["pia"],
    )

# Add edges to possible next flights
for row in df.select(["flight_id", "next_flight_ids"]).iter_rows(named=True):
    src = row["flight_id"]

    for dst in row["next_flight_ids"]:
        G.add_edge(src, dst)

In [129]:
import networkx as nx

# Keep only connected regions that include both PIA and LADD flights
pia_nodes = {
    row["flight_id"]
    for row in df.filter(pl.col("pia")).select("flight_id").iter_rows(named=True)
}

ladd_nodes = {
    row["flight_id"]
    for row in df.filter(pl.col("ladd")).select("flight_id").iter_rows(named=True)
}

# Find every node connected to a PIA node
keep_nodes = set()

for pia_node in pia_nodes:
    keep_nodes.update(nx.descendants(G, pia_node))
    keep_nodes.update(nx.ancestors(G, pia_node))
    keep_nodes.add(pia_node)

G = G.subgraph(keep_nodes).copy()

keep_components = []
for component in nx.weakly_connected_components(G):
    if component & pia_nodes and component & ladd_nodes:
        keep_components.extend(component)

# Subgraph containing only regions that can produce mixed PIA/LADD paths
G = G.subgraph(keep_components).copy()

In [130]:
df_flights.filter(pl.col("pia") == True).get_column("icao").unique()

icao
str
"""a039a2"""
"""a10100"""
"""a03969"""
"""a0fefe"""
"""a0feb1"""
…
"""a03b4c"""
"""a0ff21"""
"""a0391f"""


In [131]:
G.number_of_nodes()

60988

In [132]:
subgraphs = [
    G.subgraph(c).copy()
    for c in nx.weakly_connected_components(G)
]
len(subgraphs)

29

In [133]:
target = "a10151_2025-07-31 16:21:28"

depth = 2

nodes = set()

for node, dist in nx.single_source_shortest_path_length(
    G,
    target,
    cutoff=depth,
).items():
    nodes.add(node)

for node, dist in nx.single_source_shortest_path_length(
    G.reverse(),
    target,
    cutoff=depth,
).items():
    nodes.add(node)

G_small = G.subgraph(nodes).copy()

In [134]:
g_0 = subgraphs[0]
g_0

In [135]:
target = "a10151_2025-07-31 16:21:28"

ancestors = list(nx.ancestors(G, target))
descendants = list(nx.descendants(G, target))

paths = []

for src in ancestors + [target]:
    for dst in descendants + [target]:
        if src != dst:
            try:
                path = nx.shortest_path(G, src, dst)

                has_pia = any(G.nodes[node].get("pia") for node in path)
                has_ladd = any(G.nodes[node].get("ladd") for node in path)

                if target in path and has_pia and has_ladd:
                    paths.append(path)

                    if len(paths) >= 10:
                        break

            except nx.NetworkXNoPath:
                pass

    if len(paths) >= 2:
        break

print(paths)

[['a57878_2025-07-19 18:53:47', 'a10151_2025-07-26 19:31:31', 'a10151_2025-07-31 16:21:28'], ['a97a67_2025-07-05 06:17:36', 'a97a67_2025-07-10 21:34:05', 'a10151_2025-07-26 19:31:31', 'a10151_2025-07-31 16:21:28']]


In [136]:
from data_engineering.flights.flight_type import Flight, flights_df_to_flights
flights_objs:dict[str, Flight] = dict()
for flight in df_flights.iter_rows(named=True):
     flights_objs[flight["flight_id"]] = Flight(**flight)

In [140]:
df_path_0 = pl.DataFrame()
for flight_id in paths[1]:
    flight = flights_objs[flight_id]
    df_path_0 = pl.concat(
        [
            df_path_0,
            pl.DataFrame(
                {
                    "flight_id": flight.flight_id,
                    "icao": flight.icao,
                    "takeoff_time": flight.takeoff_time,
                    "landing_time": flight.landing_time,
                    "takeoff_airport_ident": flight.takeoff_airport_ident,
                    "landing_airport_ident": flight.landing_airport_ident,
                    "ladd": flight.ladd,
                    "pia": flight.pia,
                    "category": flight.category,
                }
            ),
        ],
        how="vertical",
    )
df_path_0

flight_id,icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,ladd,pia,category
str,str,datetime[μs],datetime[μs],str,str,bool,bool,str
"""a97a67_2025-07-05 06:17:36""","""a97a67""",2025-07-05 06:17:36.260,2025-07-05 06:25:45.130,"""KBUR""","""KVNY""",true,false,"""A7"""
"""a97a67_2025-07-10 21:34:05""","""a97a67""",2025-07-10 21:34:05.510,2025-07-10 21:51:28.760,"""KVNY""","""KCMA""",true,false,"""A7"""
"""a10151_2025-07-26 19:31:31""","""a10151""",2025-07-26 19:31:31.600,2025-07-26 19:38:44.590,"""KCMA""","""KOXR""",false,true,"""A7"""
"""a10151_2025-07-31 16:21:28""","""a10151""",2025-07-31 16:21:28.210,2025-07-31 16:29:27.760,"""KOXR""","""SZP""",false,true,"""A7"""


In [ ]:
paths[0]

['a10151_2025-07-26 19:31:31', 'a10151_2025-07-31 16:21:28']

In [ ]:
a = set(df_flights.filter(pl.col("pia") == True).get_column("icao").unique())
"a03bc7" in a

True

In [ ]:
# I should make dict with key being flight_id and value being Flight Object 

In [ ]:
# how do I deal with getting back to the icaos. 
# I can iterate for through each flight
# I'm going to try with assuuming flight data is perfect first just to see how far I get. 